# 03 - Agentic RAG

Agentic RAG with decision-making, self-correction, and multi-agent decomposition.


In [ ]:
import numpy as np
import re
import math
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import random
import warnings
warnings.filterwarnings('ignore')

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return text.split()

def cosine_similarity(v1, v2):
    dot = np.dot(v1, v2)
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return dot / (norm1 * norm2)

class TFIDF:
    def __init__(self):
        self.idf = {}
        self.vocab = []
        self.vocab_index = {}
    def fit(self, docs):
        tokenized = [preprocess(d) for d in docs]
        all_tokens = set()
        for t in tokenized: all_tokens.update(t)
        self.vocab = sorted(list(all_tokens))
        self.vocab_index = {w: i for i, w in enumerate(self.vocab)}
        N = len(docs)
        for word in self.vocab:
            df = sum(1 for tokens in tokenized if word in tokens)
            self.idf[word] = math.log(N / (df + 1)) + 1
        return self
    def transform(self, docs):
        vectors = []
        for doc in docs:
            tokens = preprocess(doc)
            tf = Counter(tokens)
            vec = np.zeros(len(self.vocab))
            for word, count in tf.items():
                if word in self.vocab_index:
                    vec[self.vocab_index[word]] = count * self.idf[word]
            norm = np.linalg.norm(vec)
            vectors.append(vec / norm if norm > 0 else vec)
        return np.array(vectors)
    def fit_transform(self, docs):
        return self.fit(docs).transform(docs)

documents = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Natural language processing allows computers to understand language.",
    "Computer vision enables machines to interpret visual information.",
    "Reinforcement learning is a type of machine learning.",
    "Supervised learning uses labeled data to train models.",
    "Unsupervised learning finds hidden patterns in data.",
    "Transfer learning allows a model to be reused.",
    "Generative AI can create new content.",
    "Large language models are trained on vast amounts of text."
]

print('Setup complete!')


## 1. Agent State & Memory


In [ ]:
class AgentMemory:
    def __init__(self, max_history=5):
        self.history = []
        self.max_history = max_history
        self.knowledge_base = {}
    def add(self, query, action, result):
        self.history.append({'query': query, 'action': action, 'result': result})
        if len(self.history) > self.max_history:
            self.history.pop(0)
    def get_context(self):
        return self.history
    def store_fact(self, key, value):
        self.knowledge_base[key] = value
    def recall(self, key):
        return self.knowledge_base.get(key, None)

memory = AgentMemory()
memory.add('What is ML?', 'search', 'Machine learning is...')
memory.store_fact('ml_definition', 'Machine learning is a subset of AI')
print(f'History: {len(memory.history)} items')
print(f'KB: {memory.knowledge_base}')


## 2. Tool Definitions


In [ ]:
def search_tool(query, corpus):
    tfidf = TFIDF()
    vectors = tfidf.fit_transform(corpus)
    q_vec = tfidf.transform([query])[0]
    scores = [(i, cosine_similarity(q_vec, v)) for i, v in enumerate(vectors)]
    scores.sort(key=lambda x: x[1], reverse=True)
    return scores[:3]

def calculate_tool(expression):
    try:
        allowed = {'sqrt': math.sqrt, 'log': math.log, 'exp': math.exp, 'sin': math.sin, 'cos': math.cos, 'pi': math.pi}
        return eval(expression, {'__builtins__': {}}, allowed)
    except:
        return None

def summarize_tool(text, max_sentences=2):
    sentences = text.split('.')
    return '. '.join(sentences[:max_sentences]) + '.'

def verify_tool(fact, kb):
    for key, value in kb.items():
        if fact.lower() in value.lower() or value.lower() in fact.lower():
            return True, value
    return False, None

def expand_tool(query):
    synonyms = {'ml': 'machine learning', 'ai': 'artificial intelligence'}
    rewritten = query.lower()
    for a, f in synonyms.items():
        rewritten = rewritten.replace(a, f)
    return rewritten

TOOLS = {'search': search_tool, 'calculate': calculate_tool, 'summarize': summarize_tool, 'verify': verify_tool, 'expand': expand_tool}
print('Tools defined!')


## 3. Agent with Decision Logic


In [ ]:
class AgenticRAG:
    def __init__(self, documents):
        self.documents = documents
        self.memory = AgentMemory()
        self.max_steps = 5
    def decide_action(self, query, step):
        query_lower = query.lower()
        if any(w in query_lower for w in ['calculate', 'compute', 'math', 'sum', 'multiply']):
            return 'calculate'
        elif step == 0:
            return 'search'
        elif step == 1 and len(query.split()) < 5:
            return 'expand'
        elif step >= 2:
            return 'verify'
        else:
            return 'search'
    def execute(self, action, query, context=''):
        if action == 'search':
            results = search_tool(query, self.documents)
            docs = [self.documents[i] for i, _ in results]
            return {'action': 'search', 'results': docs, 'scores': results}
        elif action == 'calculate':
            expr = re.search(r'[0-9+\-*/().sqrtlogexp ]+', query)
            if expr:
                result = calculate_tool(expr.group())
                return {'action': 'calculate', 'result': result}
            return {'action': 'calculate', 'result': None}
        elif action == 'expand':
            expanded = expand_tool(query)
            return {'action': 'expand', 'new_query': expanded}
        elif action == 'verify':
            ok, val = verify_tool(query, self.memory.knowledge_base)
            return {'action': 'verify', 'verified': ok, 'value': val}
        elif action == 'summarize':
            return {'action': 'summarize', 'summary': summarize_tool(context)}
        return {'action': 'unknown'}
    def run(self, query):
        print(f'Agent processing: {query}')
        current_query = query
        context = ''
        for step in range(self.max_steps):
            action = self.decide_action(current_query, step)
            print(f'  Step {step+1}: Action = {action}')
            result = self.execute(action, current_query, context)
            self.memory.add(current_query, action, result)
            if action == 'search':
                context = ' '.join(result['results'][:2])
            elif action == 'expand':
                current_query = result['new_query']
            elif action == 'calculate':
                return f'Result: {result["result"]}'
            elif action == 'verify' and result['verified']:
                return f'Verified: {result["value"]}'
            if step >= 2 and context:
                return f'Answer: {summarize_tool(context)}'
        return f'Final: {summarize_tool(context)}'

agent = AgenticRAG(documents)
print('Agent initialized!')


In [ ]:
test_queries = ['What is machine learning?', 'Calculate sqrt(144) + 5', 'Tell me about AI']
for q in test_queries:
    print(f'Query: {q}')
    response = agent.run(q)
    print(f'Response: {response}\n')


## 4. Self-Correction


In [ ]:
class SelfCorrectingAgent(AgenticRAG):
    def run(self, query):
        print(f'Self-Correcting Agent: {query}')
        current_query = query
        best_answer = None
        best_confidence = 0
        for attempt in range(3):
            print(f'  Attempt {attempt+1}:')
            answer = super().run(current_query)
            confidence = self._score_confidence(answer, current_query)
            print(f'  Confidence: {confidence:.2f}')
            if confidence > best_confidence:
                best_confidence = confidence
                best_answer = answer
            if confidence > 0.7:
                break
            current_query = self._reformulate(query, answer)
            print(f'  Reformulated: {current_query}')
        return best_answer
    def _score_confidence(self, answer, query):
        q_tokens = set(preprocess(query))
        a_tokens = set(preprocess(answer))
        overlap = len(q_tokens & a_tokens)
        return min(overlap / max(len(q_tokens), 1), 1.0)
    def _reformulate(self, original, failed_answer):
        return original + ' explain in detail'

sc_agent = SelfCorrectingAgent(documents)
result = sc_agent.run('deep learning')
print(f'Best answer: {result}')


## 5. Multi-Agent Decomposition


In [ ]:
class SpecialistAgent:
    def __init__(self, name, specialty):
        self.name = name
        self.specialty = specialty
        self.memory = []
    def can_handle(self, query):
        return any(word in query.lower() for word in self.specialty)
    def process(self, query, documents):
        relevant = [d for d in documents if any(w in d.lower() for w in self.specialty)]
        if not relevant:
            relevant = documents
        tfidf = TFIDF()
        vectors = tfidf.fit_transform(relevant)
        q_vec = tfidf.transform([query])[0]
        scores = [(i, cosine_similarity(q_vec, v)) for i, v in enumerate(vectors)]
        scores.sort(key=lambda x: x[1], reverse=True)
        best_doc = relevant[scores[0][0]] if scores else 'No relevant info'
        self.memory.append((query, best_doc))
        return f'[{self.name}] {best_doc}'

class MultiAgentSystem:
    def __init__(self, documents):
        self.documents = documents
        self.agents = [
            SpecialistAgent('ML-Expert', ['machine learning', 'deep learning', 'neural', 'training']),
            SpecialistAgent('NLP-Expert', ['language', 'nlp', 'text', 'token']),
            SpecialistAgent('Vision-Expert', ['vision', 'image', 'visual', 'computer vision']),
            SpecialistAgent('Math-Expert', ['calculate', 'math', 'compute', 'algorithm'])
        ]
    def route(self, query):
        for agent in self.agents:
            if agent.can_handle(query):
                return agent
        return self.agents[0]
    def run(self, query):
        print(f'Multi-Agent System: {query}')
        agent = self.route(query)
        print(f'  Routed to: {agent.name}')
        result = agent.process(query, self.documents)
        return result

mas = MultiAgentSystem(documents)
queries = ['How does neural network work?', 'Explain NLP concepts', 'What is computer vision?']
for q in queries:
    print(mas.run(q))
    print()


## 6. Agent Decision Visualization


In [ ]:
steps = ['Query Analysis', 'Tool Selection', 'Retrieval', 'Verification', 'Response']
confidence = [0.9, 0.85, 0.92, 0.78, 0.88]
plt.figure(figsize=(10, 5))
colors = ['#2ecc71' if c > 0.8 else '#f39c12' if c > 0.6 else '#e74c3c' for c in confidence]
bars = plt.bar(steps, confidence, color=colors, edgecolor='black')
plt.ylim(0, 1)
plt.ylabel('Confidence Score')
plt.title('Agent Decision Confidence')
plt.axhline(y=0.8, color='green', linestyle='--', alpha=0.5)
plt.axhline(y=0.6, color='orange', linestyle='--', alpha=0.5)
for bar, conf in zip(bars, confidence):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{conf:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Exercises


In [ ]:
# Exercise 1: Add planning agent for sub-queries
# Exercise 2: Implement agent negotiation
# Exercise 3: Add reflection step
# Exercise 4: Build hierarchical multi-agent system
print('Exercises listed!')
